# NB_bishop_ch13_graph_neural_networks

**Chapter 13 — Graph Neural Networks: GCN, Node Classification, and Graph Attention**

This notebook implements graph neural network components from scratch: GCN message passing, node classification on Zachary's Karate Club graph, and a graph attention mechanism.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_13/NB_bishop_ch13_graph_neural_networks.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install numpy matplotlib networkx torch scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f'PyTorch version: {torch.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## 1. GCN Message Passing from Scratch

A Graph Convolutional Network (GCN) layer computes:

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

where $\tilde{A} = A + I$ (adjacency with self-loops) and $\tilde{D}$ is the degree matrix of $\tilde{A}$.

In [ ]:
def gcn_normalize(A):
    """
    Compute the symmetric normalized adjacency: D^{-1/2} A_hat D^{-1/2}.
    
    Parameters
    ----------
    A : array (N, N) - adjacency matrix (without self-loops)
    
    Returns
    -------
    A_norm : array (N, N) - normalized adjacency with self-loops
    """
    N = A.shape[0]
    A_hat = A + np.eye(N)  # Add self-loops
    D_hat = np.diag(A_hat.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D_hat)))
    A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
    return A_norm


def gcn_layer_numpy(A_norm, H, W, activation=True):
    """
    Single GCN layer: H' = ReLU(A_norm @ H @ W).
    
    Parameters
    ----------
    A_norm : array (N, N) - normalized adjacency
    H : array (N, d_in) - node features
    W : array (d_in, d_out) - weight matrix
    activation : bool - apply ReLU
    """
    out = A_norm @ H @ W
    if activation:
        out = np.maximum(0, out)  # ReLU
    return out

# Test on a small graph
A_small = np.array([
    [0, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 0],
    [0, 1, 0, 0]
], dtype=float)

A_norm_small = gcn_normalize(A_small)
print('Adjacency matrix A:')
print(A_small)
print('\nNormalized adjacency (with self-loops):')
print(A_norm_small.round(4))

# Random node features (4 nodes, 3 features)
np.random.seed(42)
H = np.random.randn(4, 3)
W = np.random.randn(3, 2)

H_out = gcn_layer_numpy(A_norm_small, H, W)
print(f'\nInput features shape: {H.shape}')
print(f'Output features shape: {H_out.shape}')
print(f'Output:\n{H_out.round(4)}')

In [ ]:
# Demonstrate message passing: show how each node aggregates neighbor info
print('Message passing demonstration:')
print('=' * 50)
for i in range(4):
    neighbors = list(np.where(A_small[i] > 0)[0])
    weights = A_norm_small[i]
    nonzero = np.where(weights > 0)[0]
    print(f'\nNode {i}: neighbors = {neighbors}')
    print(f'  Aggregation weights (including self): ', end='')
    for j in nonzero:
        print(f'w[{i},{j}]={weights[j]:.3f}  ', end='')
    print()
    aggregated = sum(weights[j] * H[j] for j in nonzero)
    print(f'  Aggregated feature: {aggregated.round(4)}')

## 2. Node Classification on Karate Club

Zachary's Karate Club is a classic social network with 34 nodes and 2 ground-truth communities. We train a 2-layer GCN for semi-supervised node classification.

In [ ]:
# Load Karate Club graph
G = nx.karate_club_graph()
N = G.number_of_nodes()
A = nx.to_numpy_array(G)

# Ground truth labels (Mr. Hi = 0, Officer = 1)
labels = np.array([G.nodes[i]['club'] == 'Officer' for i in range(N)], dtype=int)

# Node features: one-hot identity
X = np.eye(N, dtype=np.float32)

print(f'Karate Club: {N} nodes, {G.number_of_edges()} edges')
print(f'Class 0 (Mr. Hi): {(labels == 0).sum()} nodes')
print(f'Class 1 (Officer): {(labels == 1).sum()} nodes')

In [ ]:
# PyTorch GCN for node classification
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, A_norm, H):
        return A_norm @ H @ self.weight


class GCN(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes):
        super().__init__()
        self.gcn1 = GCNLayer(n_features, n_hidden)
        self.gcn2 = GCNLayer(n_hidden, n_classes)
    
    def forward(self, A_norm, X):
        H = F.relu(self.gcn1(A_norm, X))
        H = F.dropout(H, p=0.5, training=self.training)
        H = self.gcn2(A_norm, H)
        return H

# Prepare tensors
A_norm_np = gcn_normalize(A).astype(np.float32)
A_norm_t = torch.from_numpy(A_norm_np)
X_t = torch.from_numpy(X)
y_t = torch.from_numpy(labels).long()

# Semi-supervised: only use a few labeled nodes for training
train_mask = np.zeros(N, dtype=bool)
train_mask[[0, 1, 2, 33, 32, 31]] = True  # 3 from each class
train_mask_t = torch.from_numpy(train_mask)

print(f'Training nodes: {train_mask.sum()} out of {N}')
print(f'Training labels: {labels[train_mask]}')

In [ ]:
# Train the GCN
torch.manual_seed(42)
model = GCN(n_features=N, n_hidden=16, n_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

losses = []
accuracies = []

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    
    logits = model(A_norm_t, X_t)
    loss = F.cross_entropy(logits[train_mask_t], y_t[train_mask_t])
    loss.backward()
    optimizer.step()
    
    # Evaluate on all nodes
    model.eval()
    with torch.no_grad():
        logits_eval = model(A_norm_t, X_t)
        preds = logits_eval.argmax(dim=1).numpy()
        acc = (preds == labels).mean()
    
    losses.append(loss.item())
    accuracies.append(acc)
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d}: loss={loss.item():.4f}, accuracy={acc:.4f}')

print(f'\nFinal accuracy: {accuracies[-1]:.4f}')
print(f'Predictions: {preds}')
print(f'Ground truth: {labels}')
print(f'Misclassified: {np.where(preds != labels)[0]}')

In [ ]:
# Visualize GCN embeddings and classification
model.eval()
with torch.no_grad():
    H1 = F.relu(model.gcn1(A_norm_t, X_t)).numpy()

# Use t-SNE for 2D visualization of hidden embeddings
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=10)
H_2d = tsne.fit_transform(H1)

pos = nx.spring_layout(G, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Graph with predicted labels
colors_pred = ['#1a3a6e' if p == 0 else '#cd0000' for p in preds]
nx.draw(G, pos, ax=ax1, node_color=colors_pred, node_size=300,
        with_labels=True, font_size=8, font_color='white',
        edge_color='gray', width=0.5)
ax1.set_title('GCN Predicted Labels on Karate Club')

# t-SNE of GCN embeddings
for c, name, color in [(0, 'Mr. Hi', '#1a3a6e'), (1, 'Officer', '#cd0000')]:
    mask = labels == c
    ax2.scatter(H_2d[mask, 0], H_2d[mask, 1], c=color, label=name,
               s=100, edgecolors='white', alpha=0.8)
    for i in np.where(mask)[0]:
        ax2.annotate(str(i), (H_2d[i, 0], H_2d[i, 1]), fontsize=7,
                     ha='center', va='center', color='white')
ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')
ax2.set_title('GCN Hidden Embeddings (t-SNE)')
ax2.legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch13_gcn')
plt.show()

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(losses, color='#1a3a6e', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('GCN Training Loss')

ax2.plot(accuracies, color='#228B22', linewidth=1.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('GCN Node Classification Accuracy')
ax2.set_ylim(0, 1.05)

fig.tight_layout()
save_fig(fig, 'bishop_ch13_node_classification')
plt.show()

## 3. Graph Attention Mechanism

Graph Attention Networks (GAT) replace the fixed normalization of GCN with learned attention coefficients:

$$\alpha_{ij} = \frac{\exp\left(\text{LeakyReLU}\left(\mathbf{a}^T [W h_i \| W h_j]\right)\right)}{\sum_{k \in \mathcal{N}(i)} \exp\left(\text{LeakyReLU}\left(\mathbf{a}^T [W h_i \| W h_k]\right)\right)}$$

In [ ]:
class GATLayer(nn.Module):
    """Graph Attention Layer from scratch."""
    
    def __init__(self, in_features, out_features, alpha=0.2):
        super().__init__()
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.a = nn.Parameter(torch.FloatTensor(2 * out_features, 1))
        self.leaky_relu = nn.LeakyReLU(alpha)
        
        nn.init.xavier_uniform_(self.W)
        nn.init.xavier_uniform_(self.a)
    
    def forward(self, H, A):
        """
        Parameters
        ----------
        H : tensor (N, in_features) - node features
        A : tensor (N, N) - adjacency matrix (used as mask)
        
        Returns
        -------
        H_out : tensor (N, out_features)
        attn : tensor (N, N) - attention coefficients
        """
        N = H.size(0)
        Wh = H @ self.W  # (N, out_features)
        
        # Compute attention coefficients
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)  # (N, N, out_features)
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)  # (N, N, out_features)
        concat = torch.cat([Wh_i, Wh_j], dim=-1)  # (N, N, 2*out_features)
        
        e = self.leaky_relu(concat @ self.a).squeeze(-1)  # (N, N)
        
        # Mask: only attend to neighbors (and self)
        A_mask = A + torch.eye(N)
        e = e.masked_fill(A_mask == 0, -1e9)
        
        attn = F.softmax(e, dim=-1)  # (N, N)
        H_out = attn @ Wh  # (N, out_features)
        
        return H_out, attn

# Test GAT on Karate Club
gat = GATLayer(N, 8)
A_t = torch.from_numpy(A).float()

H_gat, attn_gat = gat(X_t, A_t)
print(f'GAT input: {X_t.shape}')
print(f'GAT output: {H_gat.shape}')
print(f'Attention shape: {attn_gat.shape}')
print(f'Attention row sums: {attn_gat.sum(dim=1)[:5].detach().numpy().round(4)}')

In [ ]:
# Visualize GAT attention for selected nodes
attn_np = attn_gat.detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
highlight_nodes = [0, 33, 2]  # Key nodes

for idx, node in enumerate(highlight_nodes):
    # Get neighbors
    neighbors = list(G.neighbors(node)) + [node]
    attn_vals = attn_np[node, neighbors]
    
    # Sort by attention weight
    sort_idx = np.argsort(attn_vals)[::-1]
    sorted_neighbors = [neighbors[i] for i in sort_idx]
    sorted_attn = attn_vals[sort_idx]
    
    colors_bar = ['#cd0000' if n == node else '#1a3a6e' for n in sorted_neighbors]
    axes[idx].barh(range(len(sorted_neighbors)), sorted_attn, color=colors_bar)
    axes[idx].set_yticks(range(len(sorted_neighbors)))
    axes[idx].set_yticklabels([f'Node {n}' for n in sorted_neighbors])
    axes[idx].set_xlabel('Attention Weight')
    axes[idx].set_title(f'Node {node} Attention')
    axes[idx].invert_yaxis()

fig.suptitle('GAT Attention Coefficients', fontsize=14)
fig.tight_layout()
plt.show()

## Summary

In this notebook we explored graph neural networks from Bishop Chapter 13:
- **GCN message passing**: symmetric normalization and neighborhood aggregation
- **Node classification**: semi-supervised learning on Karate Club with only 6 labeled nodes
- **Graph attention**: learned, adaptive neighborhood weighting via GAT

In [ ]:
# Key takeaways
takeaways = [
    '1. GCN propagates information through the graph via normalized adjacency multiplication.',
    '2. Adding self-loops ensures each node retains its own features during aggregation.',
    '3. Semi-supervised GCN can classify all nodes from very few labels by exploiting graph structure.',
    '4. GAT replaces fixed normalization with learned attention, allowing adaptive neighbor weighting.',
    '5. GNNs generalize CNNs from regular grids to arbitrary graph structures.'
]
print('Key Takeaways:')
print('-' * 60)
for t in takeaways:
    print(t)